# Rekap Yield Harian Semua PV String

Notebook ini menghitung yield harian seluruh PV string yang terdeteksi pada
rentang tanggal inklusif.

- Sumber: folder Google Drive publik CSV Export PV String.
- Yield: `sum(power_kw_valid * 5/60)` tanpa mengisi data yang hilang.
- Output: workbook tiga-sheet di `output_string/`.
- Jika detail melampaui batas baris Excel, pendekkan rentang tanggal sesuai pesan error.
- Cell terakhir menyediakan download workbook saat dijalankan di Colab.

Jalankan Cell 1 sampai Cell 6 berurutan. Edit hanya tiga nilai di Cell 2.


In [ ]:
# Cell 1 - Setup Colab/repo/output
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "gdown>=6.0.0"])
from pathlib import Path
import os, tempfile

def find_repo_root(start=None):
    path = Path(start or os.getcwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pv_pipeline").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise RuntimeError("Repo root tidak ditemukan; jalankan notebook dari clone SolarYieldPro.")

PUBLIC_REPO_URL = "https://github.com/nabilhaidr/PVStringHeatmapCheck.git"
DEFAULT_REPO_DIR = Path.cwd() / "PVStringHeatmapCheck"
try:
    REPO_DIR = find_repo_root()
except RuntimeError:
    if not DEFAULT_REPO_DIR.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", PUBLIC_REPO_URL,
            str(DEFAULT_REPO_DIR),
        ])
    REPO_DIR = find_repo_root(DEFAULT_REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
OUTPUT_DIR = REPO_DIR / "output_string"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path(tempfile.mkdtemp(prefix="all_string_yield_inputs_"))
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# Cell 2 - Konfigurasi input (edit tiga nilai ini)
from pv_pipeline.string_yield_report import parse_date_range, validate_drive_folder_url
URL_CSV = "https://drive.google.com/drive/folders/1f_KrPuqfZJTE5I9cVQiyp65QrHbkF3Iw?usp=sharing"
START_DATE = "2026-05-01"
END_DATE = "2026-05-14"
DATES = parse_date_range(START_DATE, END_DATE)
validate_drive_folder_url(URL_CSV)
print(DATES[0].date(), "s.d.", DATES[-1].date())


In [ ]:
# Cell 3 - Inventaris dan download CSV selektif
from pv_pipeline.all_string_yield_report import download_csv_inputs
if any(name not in globals() for name in ("DATES", "URL_CSV", "INPUT_DIR")):
    raise RuntimeError("Jalankan Cell 2 terlebih dahulu.")
MANIFEST, INPUTS = download_csv_inputs(URL_CSV, DATES, INPUT_DIR)
print(
    "CSV inventory:", MANIFEST.csv_inventory_count,
    "dipilih:", len(MANIFEST.csv_by_date),
    "berhasil:", len(INPUTS.csv_by_date),
    "missing:", MANIFEST.missing_csv_dates,
)
print("Download errors:", INPUTS.download_errors)


In [ ]:
# Cell 4 - Hitung yield harian seluruh string
try:
    from IPython.display import display
except ImportError:
    display = print
from pv_pipeline.all_string_yield_report import build_all_string_daily_yield
if any(name not in globals() for name in ("INPUTS", "MANIFEST", "DATES")):
    raise RuntimeError("Jalankan Cell 3 terlebih dahulu.")
REPORT = build_all_string_daily_yield(
    INPUTS.csv_by_date,
    DATES,
    source_manifest=MANIFEST,
    download_errors=INPUTS.download_errors,
)
display(REPORT.summary)
STATUS_COUNTS = REPORT.daily["status"].value_counts().to_dict()
print({
    "requested_days": len(DATES),
    "detected_strings": REPORT.metadata["detected_string_count"],
    "detail_rows": len(REPORT.daily),
    "status_counts": STATUS_COUNTS,
})
print("Warnings:", REPORT.metadata["warnings"])


In [ ]:
# Cell 5 - Ekspor dan verifikasi workbook
from openpyxl import load_workbook
from pv_pipeline.all_string_yield_report import (
    build_all_string_output_path,
    verify_all_string_workbook,
    write_all_string_workbook,
)
OUTPUT_VERIFIED = False
if any(name not in globals() for name in ("REPORT", "DATES", "OUTPUT_DIR")):
    raise RuntimeError("Jalankan Cell 4 terlebih dahulu.")
OUTPUT_XLSX = build_all_string_output_path(
    OUTPUT_DIR,
    DATES[0].date(),
    DATES[-1].date(),
)
write_all_string_workbook(OUTPUT_XLSX, REPORT)
verify_all_string_workbook(OUTPUT_XLSX)
CHECK_WB = load_workbook(OUTPUT_XLSX, read_only=False, data_only=False)
print(
    "Workbook:", OUTPUT_XLSX,
    "sheets:", CHECK_WB.sheetnames,
    "bytes:", OUTPUT_XLSX.stat().st_size,
    "days:", REPORT.metadata["requested_days"],
    "strings:", REPORT.metadata["detected_string_count"],
)
CHECK_WB.close()
OUTPUT_VERIFIED = True


In [ ]:
# Cell 6 - Download dari Colab; lokal hanya menampilkan path
if (
    not globals().get("OUTPUT_VERIFIED", False)
    or "OUTPUT_XLSX" not in globals()
    or not OUTPUT_XLSX.exists()
):
    raise RuntimeError("Jalankan Cell 5 terlebih dahulu.")
try:
    from google.colab import files
except ImportError:
    print("Bukan Colab; workbook tersedia di", OUTPUT_XLSX)
else:
    files.download(str(OUTPUT_XLSX))
